# Compute Balanced Class Weights

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.utils.class_weight import compute_class_weight
import pickle
import json
import os

## Config

In [3]:
os.chdir("../../")
os.getcwd()

'c:\\Users\\Ansh Lulla\\VS-Code\\PlantDiseaseProject\\plant-disease-project'

In [4]:
DATASET_ROOT = Path('Dataset_Split_Morphological')
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

## Count Images per Class

In [5]:
class_counts = {}
train_dir = DATASET_ROOT / 'train'

for class_dir in sorted(train_dir.iterdir()):
    if class_dir.is_dir():
        images = list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.png')) + list(class_dir.glob('*.jpeg'))
        class_counts[class_dir.name] = len(images)

class_counts = dict(sorted(class_counts.items(), key=lambda x: x[1], reverse=True))
total = sum(class_counts.values())

print(f"Total classes: {len(class_counts)}")
print(f"Total images: {total}")
print(f"\nClass distribution:")
for cls, count in class_counts.items():
    print(f"  {cls}: {count}")

Total classes: 23
Total images: 28627

Class distribution:
  Tomato__Tomato_YellowLeaf__Curl_Virus: 2589
  Tomato_Bacterial_spot: 1705
  Apple___healthy: 1607
  Apple___Apple_scab: 1605
  Apple___Black_rot: 1575
  Corn_(maize)___Common_rust_: 1530
  Corn_(maize)___Northern_Leaf_Blight: 1519
  Tomato_Late_blight: 1502
  Corn_(maize)___healthy: 1473
  Apple___Cedar_apple_rust: 1467
  Tomato_Septoria_leaf_spot: 1410
  Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot: 1348
  Tomato_Spider_mites_Two_spotted_spider_mite: 1319
  Tomato_healthy: 1288
  Pepper__bell___healthy: 1223
  Tomato__Target_Spot: 1122
  Pepper__bell___Bacterial_spot: 818
  Potato___Early_blight: 792
  Potato___Late_blight: 785
  Tomato_Early_blight: 779
  Tomato_Leaf_Mold: 742
  Tomato__Tomato_mosaic_virus: 306
  Potato___healthy: 123


## Compute Balanced Class Weights

In [6]:
class_names = sorted(class_counts.keys())
class_labels = np.arange(len(class_names))

y_all = []
for cls_idx, cls_name in enumerate(class_names):
    count = class_counts[cls_name]
    y_all.extend([cls_idx] * count)

y_all = np.array(y_all)

weights = compute_class_weight('balanced', classes=class_labels, y=y_all)
class_weights = dict(zip(class_labels, weights))

print("\nClass weights (balanced):")
for cls_idx, (cls_name, weight) in enumerate(zip(class_names, weights)):
    print(f"  {cls_idx}: {cls_name:<50} = {weight:.4f}")


Class weights (balanced):
  0: Apple___Apple_scab                                 = 0.7755
  1: Apple___Black_rot                                  = 0.7903
  2: Apple___Cedar_apple_rust                           = 0.8484
  3: Apple___healthy                                    = 0.7745
  4: Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot = 0.9233
  5: Corn_(maize)___Common_rust_                        = 0.8135
  6: Corn_(maize)___Northern_Leaf_Blight                = 0.8194
  7: Corn_(maize)___healthy                             = 0.8450
  8: Pepper__bell___Bacterial_spot                      = 1.5216
  9: Pepper__bell___healthy                             = 1.0177
  10: Potato___Early_blight                              = 1.5715
  11: Potato___Late_blight                               = 1.5855
  12: Potato___healthy                                   = 10.1191
  13: Tomato_Bacterial_spot                              = 0.7300
  14: Tomato_Early_blight                                =

## Save Class Weights

In [7]:
weights_dict = {
    'class_names': class_names,
    'class_weights': class_weights,
    'class_counts': class_counts
}

with open(OUTPUT_DIR / 'class_weights.json', 'w') as f:
    json.dump({str(k): v for k, v in class_weights.items()}, f, indent=2)

with open(OUTPUT_DIR / 'class_names.json', 'w') as f:
    json.dump(class_names, f, indent=2)

with open(OUTPUT_DIR / 'class_weights.pkl', 'wb') as f:
    pickle.dump(class_weights, f)

print(f"Saved to {OUTPUT_DIR}:")
print(f"  - class_weights.json")
print(f"  - class_weights.pkl")
print(f"  - class_names.json")

Saved to outputs:
  - class_weights.json
  - class_weights.pkl
  - class_names.json
